In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config

In [ ]:
lake = 'geneva'
model = f'{lake}_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "eof", "synthetic_events")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC']) + 1) * grid_resolution - grid_resolution / 2
ds['XC'] = np.arange(1, len(ds['XC']) + 1) * grid_resolution - grid_resolution / 2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

In [ ]:
mask = ds['THETA'].isel(time=0).values > 0

In [ ]:
profile_xc_idx = 400
profile_yc_idx = 160

t=-1
plt.close('all')
plt.figure(figsize=(20,4))
ds_sel = ds['THETA'].isel(Z=0, time=t)
ds_sel.where(ds_sel != 0, np.nan).plot()
plt.scatter(ds.XC.isel(XC=profile_xc_idx), ds.YG.isel(YG=profile_yc_idx), c='r', s=50, marker='x')
plt.title(ds_sel.time.values)
plt.show()

In [ ]:
profile = ds.THETA.isel(XC=profile_xc_idx, YC=profile_yc_idx)

In [ ]:
import pandas as pd
import calendar

year = 2025


def _mean_profile_for_period(m: int, start_day: int, end_day: int) -> xr.DataArray:
    last_day = calendar.monthrange(year, m)[1]
    end_day_eff = min(end_day, last_day)
    start = f"{year}-{m:02}-{start_day:02}T00:00:00"
    end = f"{year}-{m:02}-{end_day_eff:02}T23:59:59"
    mp = profile.sel(time=slice(start, end)).mean(dim=["time"])
    mp = mp.where(mp != 0, np.nan).ffill(dim="Z")
    return mp


profiles = {}
for m in range(1, 13):
    da_1_15 = _mean_profile_for_period(m, 1, 15)
    da_15_31 = _mean_profile_for_period(m, 15, 31)
    profiles[f"{year}-{m:02}-07"] = da_1_15
    profiles[f"{year}-{m:02}-22"] = da_15_31

df_profiles = pd.DataFrame({k: v.values for k, v in profiles.items()}, index=da_1_15["Z"].values)
df_profiles.index.name = "Z"

csv_path = os.path.join(output_folder, f"mean_profiles_{year}_bimonthly.csv")

In [ ]:
# Save CSV
df_profiles.to_csv(csv_path)